# ISLR Phase 2 — Isolated Sign Language Recognition (20 Glosses)

Pipeline nhận diện ngôn ngữ ký hiệu độc lập người ký (Signer-Independent) sử dụng bộ dữ liệu Kaggle `asl-signs`.

**Cải tiến Schema v2 & Cấu hình T4 Balanced:**
- **Schema v2 Features**: 160 chiều (`hands_pose`), bounded interpolation (<= 3 frames), endpoint-gated velocity, centered pose reflection.
- **Thin Notebook**: Không chứa business logic, chỉ điều phối qua các CLI scripts chuẩn hóa.
- **T4 Balanced Config**: Batch size 128, `preload: device`, tối đa 80 epoch, `early_stopping_min_epochs: 15`.
- **Zero data leakage**: Split theo `participant_id`, kiểm tra nghiêm ngặt không rò rỉ người ký.
- **Đầy đủ kiểm chứng**: Sanity Overfit check (acc >= 95%), Random Forest baseline, và GRU multi-seed.

## 1. Setup Môi trường & Đồng bộ Code từ GitHub (`main`)

In [ ]:
import os
import sys
from pathlib import Path

# 1. Nhận diện môi trường
in_colab = 'google.colab' in sys.modules
in_kaggle = 'kaggle_web_client' in sys.modules or os.path.exists('/kaggle/working')

if in_colab or in_kaggle:
    env_label = "Google Colab" if in_colab else "Kaggle Notebook"
    print(f"==> Môi trường: {env_label}")
    repo_name = "SignLanguageReconition_ML"
    base_work = Path("/kaggle/working" if in_kaggle else "/content")
    repo_path = base_work / repo_name

    # Clone nếu chưa có, hoặc pull bản mới nhất từ nhánh main
    if not repo_path.is_dir():
        !cd {base_work} && git clone -b main https://github.com/ChienPM-27/SignLanguageReconition_ML.git
    else:
        print("==> Đã có thư mục repo. Đang đồng bộ code mới nhất từ nhánh main...")
        !cd {repo_path} && git fetch origin && git checkout main && git pull origin main

    %cd {repo_path}
    !pip install -q -r requirements.txt

    # Nạp Kaggle Credentials từ Secrets nếu có
    if in_colab:
        try:
            from google.colab import userdata
            try:
                os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
                print("✓ Đã nạp KAGGLE_API_TOKEN từ Colab Secrets.")
            except Exception:
                os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
                os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
                print("✓ Đã nạp KAGGLE_USERNAME và KAGGLE_KEY từ Colab Secrets.")
        except Exception as e:
            print(f"! Lưu ý Colab Secrets: {e}")
    elif in_kaggle:
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            try:
                os.environ["KAGGLE_API_TOKEN"] = user_secrets.get_secret("KAGGLE_API_TOKEN")
                print("✓ Đã nạp KAGGLE_API_TOKEN từ Kaggle Secrets.")
            except Exception:
                os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
                os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
                print("✓ Đã nạp KAGGLE_USERNAME và KAGGLE_KEY từ Kaggle Secrets.")
        except Exception as e:
            print(f"! Lưu ý Kaggle Secrets: {e}")
else:
    print("==> Môi trường: Local")

# Đảm bảo repo root nằm trong sys.path
repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Working Directory: {repo_root}")

## 2. Cấu hình Thực nghiệm (Experimental Preset & Overrides)

Chọn cấu hình huấn luyện mong muốn:
- `"t4_balanced"` (Khuyến nghị): batch 128, epochs 80, min_epochs 15, seed 42, preload device.
- `"t4_regularized"`: Thêm dropout 0.4 và weight decay 0.001 chống overfitting.
- `"smoke_test"`: Chạy nhanh 3 files/gloss kiểm thử pipeline.

In [ ]:
# Lựa chọn chế độ: 't4_balanced', 't4_regularized', hoặc 'smoke_test'
EXPERIMENT_PRESET = "t4_balanced"

OVERRIDES = []

if EXPERIMENT_PRESET == "t4_balanced":
    print("==> Áp dụng Cấu hình T4 Balanced (GRU 128x2, batch 128, max 80 epochs)")
    OVERRIDES.extend([
        "run.name=gru_t4_balanced",
        "training.device=cuda",
        "training.preload=device",
        "training.batch_size=128",
        "training.epochs=80",
        "training.early_stopping_min_epochs=15",
        "training.early_stopping_min_delta=0.0001",
        "training.amp=false",
        "training.num_seeds=1",
        "training.seeds=[42]",
    ])
elif EXPERIMENT_PRESET == "t4_regularized":
    print("==> Áp dụng Cấu hình Regularized (Dropout 0.4, Weight Decay 0.001)")
    OVERRIDES.extend([
        "run.name=gru_t4_regularized",
        "training.device=cuda",
        "training.preload=device",
        "training.batch_size=128",
        "training.epochs=80",
        "training.early_stopping_min_epochs=15",
        "model.dropout=0.4",
        "training.weight_decay=0.001",
        "training.amp=false",
        "training.num_seeds=1",
        "training.seeds=[42]",
    ])
elif EXPERIMENT_PRESET == "smoke_test":
    print("⚠️ CẢNH BÁO: Smoke test: split không phủ đủ gloss, chỉ kiểm tra pipeline chạy được.")
    OVERRIDES.extend([
        "data.max_files_per_gloss=3",
        "training.epochs=15",
        "training.patience=5",
        "training.seeds=[42]",
    ])

# Chuyển thành chuỗi cờ CLI
OVERRIDE_FLAGS = " ".join(f"--set {opt}" for opt in OVERRIDES)
print(f"CLI Overrides: {OVERRIDE_FLAGS}")

## 3. Step 0: Kiểm tra Kết nối & Cấu hình (Setup Check)

In [ ]:
!python scripts/00_setup_check.py --config configs/islr.yaml {OVERRIDE_FLAGS}

## 4. Step 1-4: Pipeline Dữ liệu Schema v2 (Selection -> Preprocess -> Split)

Trích xuất 20 glosses, sinh đặc trưng 160 chiều (`hands_pose`) theo Schema v2 và chia tập độc lập người ký.

In [ ]:
# Chạy toàn bộ pipeline chuẩn bị dữ liệu (tự động đọc dataset có sẵn tại /kaggle/input/asl-signs)
!python scripts/10_prepare_data.py --config configs/islr.yaml --stage all {OVERRIDE_FLAGS}

## 5. Step 5a: Sanity Check — Overfit Verification

Huấn luyện 1 mẫu/gloss với dropout=0 và tắt augmentation; kỳ vọng accuracy đạt >= 95% sau vài chục epoch.

In [ ]:
!python scripts/20_train.py --config configs/islr.yaml --overfit-check {OVERRIDE_FLAGS}

## 6. Step 5b: Baseline Thống kê — Random Forest

Huấn luyện Random Forest trên đặc trưng thống kê (mean, std, min, max qua trục thời gian) làm mốc so sánh cổ điển.

In [ ]:
!python scripts/20_train.py --config configs/islr.yaml --baseline-rf {OVERRIDE_FLAGS}

## 7. Step 5c: Huấn luyện Mô hình GRU T4 (Config-driven)

Chạy huấn luyện chính thức với cấu hình T4 đề xuất (`batch_size=128`, `preload=device`, tối đa 80 epoch).

In [ ]:
!python scripts/20_train.py --config configs/islr.yaml {OVERRIDE_FLAGS}

## 8. Hiển thị Kết quả & Biểu đồ Learning Curves

In [ ]:
from IPython.display import Image, display
from pathlib import Path

runs_dir = Path('outputs/runs')
if not runs_dir.is_dir():
    if Path('/kaggle/working/outputs/runs').is_dir():
        runs_dir = Path('/kaggle/working/outputs/runs')
    elif Path('../outputs/runs').is_dir():
        runs_dir = Path('../outputs/runs')

if runs_dir.is_dir():
    recent_runs = sorted(runs_dir.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    if recent_runs:
        latest_run = recent_runs[0]
        print(f'==> Hiển thị kết quả từ: {latest_run.name}')
        
        # Hiển thị biểu đồ learning curve
        for img in latest_run.rglob('*.png'):
            print(f'Ảnh: {img.name}')
            display(Image(filename=str(img)))
            
        # In tóm tắt báo cáo JSON
        for rep in latest_run.rglob('*.json'):
            print(f'\nBáo cáo {rep.name}:')
            content = rep.read_text(encoding='utf-8')
            print(content[:1500])
else:
    print('Chưa có thư mục runs trong outputs/runs')

## 9. Đóng gói & Tải về Kết quả (Artifacts Zip)

In [ ]:
from pathlib import Path

archive_name = 'islr_phase2_artifacts'
base_dir = '/kaggle/working' if Path('/kaggle/working/outputs').is_dir() else '.'
!cd {base_dir} && zip -q -r {archive_name}.zip outputs/ reports/ data/selection/ 2>/dev/null || tar -czf {archive_name}.tar.gz -C {base_dir} outputs reports data/selection

print('✓ Đã tạo file nén kết quả. Bạn có thể tải file từ file browser bên phải:')
for ext in ['.zip', '.tar.gz']:
    for p in [Path(f'{archive_name}{ext}'), Path(f'/kaggle/working/{archive_name}{ext}')]:
        if p.is_file():
            print(f'- {p.resolve()} ({p.stat().st_size / 1024:.1f} KB)')
            break